# Robo-Greeno Hexapod — Kinematics Demos

Interactive MuJoCo demos of a six-legged **PhantomX-class hexapod**, built for the Robo-Greeno OJT (Data A, Stage A).

Everything is combined into this one notebook: the robot's geometry, the closed-form inverse kinematics, the MuJoCo model, and two demos — a **pose & wave** routine and the **tripod walk**.

**How to run it:** `Runtime → Run all`. The two demos render as videos at the bottom of the notebook. A standard CPU Colab runtime is fine — no GPU needed.

## Setup
Install MuJoCo (the physics engine) and `mediapy` (to show videos inline).

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # offscreen rendering backend (works on Colab)

import math
import mujoco
import mediapy as media

print("MuJoCo", mujoco.__version__, "- ready")

## 1 · The robot's geometry

This is the *one config*: every length, angle and limit that describes the robot. The model and the kinematics below both read from these names — so adapting to real hardware later means editing only this cell.

In [ ]:
# Every number describing the robot lives here -- the "one config".
# Units: metres and radians.  Frame: +X forward, +Y left, +Z up.

# Leg link lengths (coxa -> femur -> tibia), 4 : 8 : 13 proportions
COXA, FEMUR, TIBIA = 0.040, 0.080, 0.130
FOOT_RADIUS = 0.012                       # rounded foot / ground contact

# Body
BODY_RADIUS, BODY_HALF_H, TRUNK_MASS = 0.100, 0.018, 0.45

# The six legs: (name, mount angle).  0 deg = forward, +CCW.
LEGS = [("front_left",  math.radians(  45)),
        ("mid_left",    math.radians(  90)),
        ("back_left",   math.radians( 135)),
        ("back_right",  math.radians(-135)),
        ("mid_right",   math.radians( -90)),
        ("front_right", math.radians( -45))]

# Alternating tripod gait groups (indices into LEGS)
TRIPOD_A, TRIPOD_B = [0, 2, 4], [1, 3, 5]

# Joint travel limits (radians)
COXA_RANGE  = (math.radians(-50),  math.radians( 50))
FEMUR_RANGE = (math.radians(-90),  math.radians(120))
TIBIA_RANGE = (math.radians(-170), math.radians( 20))

# Standing stance and tripod-walk settings
STANCE_RADIUS, STANCE_HEIGHT = 0.200, 0.075
GAIT_PERIOD, GAIT_STRIDE, GAIT_LIFT = 1.4, 0.060, 0.030

print(f"{len(LEGS)} legs x 3 joints = {3*len(LEGS)} joints configured")

## 2 · Inverse kinematics

A leg has three joints: **coxa** (yaw), **femur** and **tibia** (pitch). Inverse kinematics answers: *given where we want the foot, what three joint angles put it there?* It is exact, closed-form trigonometry — `leg_fk` undoes `leg_ik`, which the round-trip check below confirms.

In [ ]:
# Closed-form kinematics for one 3-DOF leg -- pure trigonometry,
# no solver, no iteration.

def clamp(v, lo=-1.0, hi=1.0):
    return max(lo, min(hi, v))

def leg_ik(x, y, z, L1=COXA, L2=FEMUR, L3=TIBIA):
    """Foot target (leg frame) -> (coxa, femur, tibia) angles, or None."""
    coxa = math.atan2(y, x)
    r    = math.hypot(x, y)
    rho  = r - L1
    D    = math.hypot(rho, z)
    if not (abs(L2 - L3) - 1e-9 <= D <= L2 + L3 + 1e-9):
        return None                            # out of reach
    knee  = math.acos(clamp((L2*L2 + L3*L3 - D*D) / (2*L2*L3)))
    beta  = math.acos(clamp((L2*L2 + D*D - L3*L3) / (2*L2*D)))
    return (coxa, math.atan2(z, rho) + beta, knee - math.pi)

def leg_fk(coxa, femur, tibia, L1=COXA, L2=FEMUR, L3=TIBIA):
    """(coxa, femur, tibia) angles -> foot (x, y, z). Inverse of leg_ik."""
    pt  = femur + tibia
    rho = L1 + L2*math.cos(femur) + L3*math.cos(pt)
    return (rho*math.cos(coxa), rho*math.sin(coxa),
            L2*math.sin(femur) + L3*math.sin(pt))

def body_target_to_leg(foot, mount, body_radius=BODY_RADIUS):
    """Rotate a body-frame foot target into one leg's own frame."""
    fx, fy, fz = foot
    dx = fx - body_radius*math.cos(mount)
    dy = fy - body_radius*math.sin(mount)
    c, s = math.cos(mount), math.sin(mount)
    return (dx*c + dy*s, -dx*s + dy*c, fz)

def solve_all(targets):
    """Solve all six legs. targets: 6 body-frame (x,y,z) foot points."""
    out = []
    for (name, mount), tgt in zip(LEGS, targets):
        sol = leg_ik(*body_target_to_leg(tgt, mount))
        if sol is None:
            raise ValueError(f"leg '{name}' cannot reach {tgt}")
        out.append(sol)
    return out

def default_stance(stance_radius=None, stance_height=None):
    """Six foot targets (body frame) for a neutral stand."""
    R = STANCE_RADIUS if stance_radius is None else stance_radius
    H = STANCE_HEIGHT if stance_height is None else stance_height
    fz = FOOT_RADIUS - H
    return [(R*math.cos(m), R*math.sin(m), fz) for _, m in LEGS]

# sanity check: forward kinematics should undo inverse kinematics
_t = default_stance()[0]
_leg_xyz = body_target_to_leg(_t, LEGS[0][1])
_ang = leg_ik(*_leg_xyz)
_err = math.dist(_leg_xyz, leg_fk(*_ang))
print("leg 0 angles (deg):",
      tuple(round(math.degrees(a), 1) for a in _ang),
      "| FK round-trip error:", f"{_err:.1e} m")

## 3 · Build the MuJoCo robot

`build_mjcf()` writes the robot as MJCF (MuJoCo's XML) straight from the config: a round trunk, six identical 3-DOF legs, 18 hinge joints, one position servo per joint, and a ground plane.

In [ ]:
# Build the MuJoCo model (MJCF / XML) straight from the config above.

def _leg(name, mount):
    rx, ry = BODY_RADIUS*math.cos(mount), BODY_RADIUS*math.sin(mount)
    c0, c1 = COXA_RANGE
    f0, f1 = FEMUR_RANGE
    t0, t1 = TIBIA_RANGE
    return f"""
      <body name="{name}_coxa" pos="{rx:.6f} {ry:.6f} 0" euler="0 0 {mount:.6f}">
        <joint name="{name}_coxa" axis="0 0 1" range="{c0:.6f} {c1:.6f}"/>
        <geom type="capsule" fromto="0 0 0 {COXA:.6f} 0 0" size="0.012" rgba="0.60 0.58 0.54 1"/>
        <body name="{name}_femur" pos="{COXA:.6f} 0 0">
          <joint name="{name}_femur" axis="0 -1 0" range="{f0:.6f} {f1:.6f}"/>
          <geom type="capsule" fromto="0 0 0 {FEMUR:.6f} 0 0" size="0.010" rgba="0.11 0.62 0.46 1"/>
          <body name="{name}_tibia" pos="{FEMUR:.6f} 0 0">
            <joint name="{name}_tibia" axis="0 -1 0" range="{t0:.6f} {t1:.6f}"/>
            <geom type="capsule" fromto="0 0 0 {TIBIA:.6f} 0 0" size="0.008" rgba="0.18 0.49 0.85 1"/>
            <geom name="{name}_foot" type="sphere" pos="{TIBIA:.6f} 0 0" size="{FOOT_RADIUS:.6f}" rgba="0.85 0.35 0.19 1"/>
          </body>
        </body>
      </body>"""

def _actuators():
    rows = []
    for name, _ in LEGS:
        for j, rng in (("coxa", COXA_RANGE), ("femur", FEMUR_RANGE), ("tibia", TIBIA_RANGE)):
            kp = 18.0 if j == "coxa" else 30.0
            rows.append(f'    <position name="{name}_{j}" joint="{name}_{j}" '
                        f'kp="{kp}" ctrlrange="{rng[0]:.6f} {rng[1]:.6f}"/>')
    return "\n".join(rows)

def build_mjcf():
    legs = "".join(_leg(n, m) for n, m in LEGS)
    return f"""<mujoco model="robo_greeno_hexapod">
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.002" integrator="implicitfast" gravity="0 0 -9.81"/>
  <default>
    <joint damping="0.14" armature="0.012"/>
    <geom friction="1.1 0.06 0.01" density="700"/>
  </default>
  <visual>
    <headlight diffuse="0.5 0.5 0.5" ambient="0.4 0.4 0.4"/>
    <global offwidth="1280" offheight="960"/>
  </visual>
  <worldbody>
    <light pos="0 0 1.4" dir="0 0 -1" diffuse="0.7 0.7 0.7"/>
    <geom name="ground" type="plane" size="3 3 0.1" rgba="0.92 0.91 0.87 1"/>
    <body name="trunk" pos="0 0 {STANCE_HEIGHT:.4f}">
      <freejoint name="trunk"/>
      <geom name="trunk" type="cylinder" size="{BODY_RADIUS:.4f} {BODY_HALF_H:.4f}"
            mass="{TRUNK_MASS}" rgba="0.36 0.35 0.33 1"/>{legs}
    </body>
  </worldbody>
  <actuator>
{_actuators()}
  </actuator>
</mujoco>
"""

_m = mujoco.MjModel.from_xml_string(build_mjcf())
print(f"MuJoCo model built: {_m.njnt} joints, {_m.nu} servos, {_m.nbody} bodies")

## 4 · Simulation and rendering helpers

`command()` is the heart of it: take six foot targets, solve the IK, write the 18 servos. `render_demo()` runs the robot forward in time and captures a video with a camera that tracks the body.

In [ ]:
# Simulation helpers + a renderer that turns a demo into a video.

def make_sim():
    m = mujoco.MjModel.from_xml_string(build_mjcf())
    return m, mujoco.MjData(m)

def _aid(m, name):
    return mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, name)

def _jadr(m, name):
    return m.jnt_qposadr[mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, name)]

def init_stance(m, d):
    """Start the robot already standing so it does not snap on spawn."""
    mujoco.mj_resetData(m, d)
    t = _jadr(m, "trunk")
    d.qpos[t:t+7] = [0, 0, STANCE_HEIGHT, 1, 0, 0, 0]
    for (name, mount), tgt in zip(LEGS, default_stance()):
        coxa, femur, tibia = leg_ik(*body_target_to_leg(tgt, mount))
        for j, v in (("coxa", coxa), ("femur", femur), ("tibia", tibia)):
            d.qpos[_jadr(m, f"{name}_{j}")] = v
    mujoco.mj_forward(m, d)

def command(m, d, foot_targets):
    """Solve the IK for six foot targets and write the eighteen servos."""
    for (name, _), (coxa, femur, tibia) in zip(LEGS, solve_all(foot_targets)):
        d.ctrl[_aid(m, f"{name}_coxa")]  = coxa
        d.ctrl[_aid(m, f"{name}_femur")] = femur
        d.ctrl[_aid(m, f"{name}_tibia")] = tibia

def render_demo(target_fn, seconds, fps=30,
                azimuth=135, elevation=-18, distance=0.95):
    """Run the robot, following target_fn(time), and capture a video."""
    m, d = make_sim()
    init_stance(m, d)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
    cam.trackbodyid = m.body("trunk").id
    cam.distance, cam.azimuth, cam.elevation = distance, azimuth, elevation
    renderer = mujoco.Renderer(m, height=480, width=640)
    frames = []
    every = max(1, round(1.0 / fps / m.opt.timestep))
    steps = int(seconds / m.opt.timestep)
    for i in range(steps):
        command(m, d, target_fn(d.time))
        mujoco.mj_step(m, d)
        if i % every == 0:
            renderer.update_scene(d, camera=cam)
            frames.append(renderer.render())
    renderer.close()
    print(f"rendered {len(frames)} frames ({seconds} s)")
    return frames

print("helpers ready: make_sim, init_stance, command, render_demo")

## Demo 1 · Pose & wave

The robot stands, crouches low, rises up tall, then lifts its front-left leg and waves it side to side. No gait — every pose is just six foot targets handed to the inverse kinematics.

In [ ]:
# Five legs hold a stand; the front-left leg lifts high and waves.
WAVE_LEG = 0          # index into LEGS  ->  "front_left"

def wave_pose(t):
    targets = default_stance()                  # all six on the ground
    _, mount = LEGS[WAVE_LEG]
    sweep = 0.060 * math.sin(2.0 * math.pi * t / 1.4)   # side to side
    targets[WAVE_LEG] = (0.21 * math.cos(mount),
                         0.21 * math.sin(mount) + sweep,
                         -0.048)                # lifted off the ground
    return targets

def pose_at(t):
    """Scripted routine: stand, crouch low, stand tall, wave."""
    t = t % 18.0
    if t < 4.0:
        return default_stance()
    if t < 8.0:
        return default_stance(stance_height=STANCE_HEIGHT * 0.70)
    if t < 12.0:
        return default_stance(stance_height=STANCE_HEIGHT * 1.12,
                              stance_radius=STANCE_RADIUS * 0.94)
    return wave_pose(t - 12.0)

frames = render_demo(pose_at, seconds=18)
media.show_video(frames, fps=30, title="Demo 1 - pose & wave")

## Demo 2 · Tripod walk

The robot walks forward with an alternating tripod gait: three legs (tripod A) push the body while the other three (tripod B) lift and swing ahead — then they swap. One tripod is always on the ground holding the robot up.

In [ ]:
# Alternating tripod gait: three legs push while three swing, then swap.
def walk_targets(t):
    base = default_stance()
    half = GAIT_STRIDE / 2.0
    out = []
    for i, (name, mount) in enumerate(LEGS):
        bx, by, bz = base[i]
        phase = (t / GAIT_PERIOD) % 1.0
        # tripod A leads; tripod B is half a cycle behind
        local = phase if i in TRIPOD_A else (phase + 0.5) % 1.0
        if local < 0.5:                         # swing: lift, carry forward
            s = local / 0.5
            dx = -half + s * GAIT_STRIDE
            dz = GAIT_LIFT * math.sin(math.pi * s)
        else:                                   # stance: push the body
            s = (local - 0.5) / 0.5
            dx = half - s * GAIT_STRIDE
            dz = 0.0
        out.append((bx + dx, by, bz + dz))
    return out

frames = render_demo(walk_targets, seconds=12, azimuth=150)
media.show_video(frames, fps=30, title="Demo 2 - tripod walk")

## Make it yours

A few things to try by editing the cells above and re-running:

- **Pose & wave** — change `WAVE_LEG` (0–5) to wave a different leg; add a new pose to `pose_at()`; make two legs wave at once.
- **Tripod walk** — change `GAIT_PERIOD`, `GAIT_STRIDE` or `GAIT_LIFT` in the config cell to walk faster, take longer steps, or lift the feet higher.
- **Harder** — give the left and right legs different stride lengths so the robot turns; or make it walk backward.

The interactive 3D viewer (`python run.py`) and this notebook run the *same* kinematics — this is the Robo-Greeno Stage A milestone.